# TALN — TP 2 : Prétraitement – Vectorisation des mots
**FST — IDS 4 / IGL 5 — 2026-2027**

Fichiers attendus dans le même dossier que ce notebook :
- `Cleaned_Indian_Food_Dataset.csv` (à télécharger depuis Kaggle : *Indian Recipes Dataset*) — Partie II, Exercice 1
- `alice.txt` **ou** `alice.docx` — Partie II, Exercice 2

## 0. Installation et imports

In [ ]:
# À exécuter une seule fois (décommenter si besoin)
# !pip install pandas scikit-learn nltk gensim python-docx
# !pip install fasttext        # Linux / macOS / Colab
# !pip install fasttext-wheel  # alternative si 'fasttext' ne s'installe pas (Windows)

In [ ]:
import re
import os
import unicodedata

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

for res in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(res, quiet=True)

# Partie I — Bag of Words sur les revues Spider-Man

### Q1. Création des documents et prétraitement
Étapes : minuscules → suppression des accents → suppression de la ponctuation → tokenisation.

In [ ]:
revue1 = "Spider man est un film incroyable !"
revue2 = "Spider man est le meilleur film !"
revue3 = "Spiderman est TELLEMENT génial."
revues = [revue1, revue2, revue3]

def pretraiter(texte):
    texte = texte.lower()                                            # minuscules
    texte = unicodedata.normalize("NFD", texte)                      # génial -> genial
    texte = "".join(c for c in texte if unicodedata.category(c) != "Mn")
    texte = re.sub(r"[^a-z0-9\s]", " ", texte)                       # ponctuation
    return texte.split()                                             # tokenisation

docs = [pretraiter(r) for r in revues]
for i, d in enumerate(docs, 1):
    print(f"Document {i} :", d)

vocabulaire = set(docs[0]).union(*docs[1:])
print("\nVocabulaire :", vocabulaire)
print("Taille :", len(vocabulaire))

In [ ]:
# Vérification par rapport au vocabulaire attendu dans l'énoncé
attendu = {'man', 'est', 'incroyable', 'meilleur', 'spiderman', 'spider',
           'genial', 'le', 'tellement', 'film', 'un'}
print("Vocabulaire conforme :", vocabulaire == attendu)

### Q2. Fonction `calculBOW`

In [ ]:
def calculBOW(document, vocabulaire):
    """Retourne un dictionnaire {mot: nombre d'occurrences} du document
    pour chaque mot du vocabulaire."""
    bow = dict.fromkeys(vocabulaire, 0)
    for mot in document:
        if mot in bow:
            bow[mot] += 1
    return bow

### Q3. Application sur les trois documents

In [ ]:
vocab_trie = sorted(vocabulaire)
bows = [calculBOW(d, vocab_trie) for d in docs]
df_bow = pd.DataFrame(bows, columns=vocab_trie, index=["Revue 1", "Revue 2", "Revue 3"])
df_bow

### Q4. Comparaison avec `sklearn` (`CountVectorizer`)

In [ ]:
corpus = [" ".join(d) for d in docs]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
df_sk = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(),
                     index=["Revue 1", "Revue 2", "Revue 3"])
df_sk

In [ ]:
print("Même résultat que calculBOW :", df_sk.equals(df_bow[df_sk.columns]))

**Remarque :** `CountVectorizer` fait lui-même la mise en minuscules et la tokenisation (motif `\b\w\w+\b`, qui ignore les mots d'une seule lettre).
En lui passant `strip_accents='unicode'`, il pourrait aussi traiter directement les revues brutes.

### Q5. Suppression des mots vides *sans ajouter de ligne de code*
On garde exactement la même cellule que la Q4 : on modifie seulement l'appel `CountVectorizer(...)` en ajoutant le paramètre `stop_words` (liste des mots vides français de NLTK).

In [ ]:
vectorizer = CountVectorizer(stop_words=stopwords.words("french"))
X = vectorizer.fit_transform(corpus)
df_sk = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(),
                     index=["Revue 1", "Revue 2", "Revue 3"])
df_sk

Les mots vides `est`, `le` et `un` ont disparu du vocabulaire.

### Q6. Uni-grammes + Bi-grammes

In [ ]:
vectorizer = CountVectorizer(stop_words=stopwords.words("french"), ngram_range=(1, 2))
X = vectorizer.fit_transform(corpus)
df_ngram = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(),
                        index=["Revue 1", "Revue 2", "Revue 3"])
df_ngram

In [ ]:
# Variante : uni + bi-grammes sans suppression des mots vides
vectorizer_all = CountVectorizer(ngram_range=(1, 2))
X_all = vectorizer_all.fit_transform(corpus)
pd.DataFrame(X_all.toarray(), columns=vectorizer_all.get_feature_names_out(),
             index=["Revue 1", "Revue 2", "Revue 3"])

# Partie II
## Exercice 1 — FastText sur *Indian Recipes Dataset*

### Q1. Téléchargement
Télécharger `Cleaned_Indian_Food_Dataset.csv` depuis Kaggle (*Indian Recipes Dataset*) et le placer à côté de ce notebook.

In [ ]:
import fasttext

CSV_PATH = "Cleaned_Indian_Food_Dataset.csv"
assert os.path.exists(CSV_PATH), f"Fichier introuvable : placez {CSV_PATH} dans {os.getcwd()}"

### Q2. Chargement dans un DataFrame

In [ ]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df.columns.tolist())
df.head()

### Q3. Une instruction traduite avant prétraitement

In [ ]:
df = df.dropna(subset=["TranslatedInstructions"]).reset_index(drop=True)
print(df.loc[0, "TranslatedRecipeName"], "\n")
print(df.loc[0, "TranslatedInstructions"])

### Q4. Nettoyage des instructions

In [ ]:
def nettoyer(texte):
    texte = str(texte).lower()                    # minuscules
    texte = re.sub(r"[^a-z\s]", " ", texte)       # supprime chiffres, ponctuation, caractères spéciaux
    texte = re.sub(r"\s+", " ", texte).strip()    # espaces multiples
    return texte

df["clean_instructions"] = df["TranslatedInstructions"].apply(nettoyer)
df = df[df["clean_instructions"].str.len() > 0]
print(df["clean_instructions"].iloc[0])

### Q5. Sauvegarde dans `food_recipes.txt` (sans en-tête ni index)

In [ ]:
df["clean_instructions"].to_csv("food_recipes.txt", header=False, index=False)

with open("food_recipes.txt", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline()[:150])

### Q6. Entraînement avec `train_unsupervised`

In [ ]:
model_ft = fasttext.train_unsupervised(
    "food_recipes.txt",
    model="skipgram",   # ou "cbow"
    dim=100,
    epoch=10,
    minCount=2,
    minn=3, maxn=6      # sous-mots (n-grammes de caractères)
)
print("Taille du vocabulaire :", len(model_ft.words))

### Q7. Mots les plus proches

In [ ]:
for mot in ["paneer", "spicy", "rice", "masala"]:
    print(f"\n=== {mot} ===")
    for score, voisin in model_ft.get_nearest_neighbors(mot, k=10):
        print(f"  {voisin:<20} {score:.4f}")

On observe que les voisins sont des ingrédients / plats proches (ex. *paneer* → autres fromages ou légumes utilisés en curry, *rice* → *basmati*, *pulao*…) : FastText capture bien le contexte culinaire.

### Q8. Mots légèrement modifiés (hors vocabulaire)

In [ ]:
for mot in ["paneeer", "masalaa", "ricee", "spicyy"]:
    print(f"\n=== {mot} (dans le vocabulaire : {mot in model_ft.words}) ===")
    for score, voisin in model_ft.get_nearest_neighbors(mot, k=5):
        print(f"  {voisin:<20} {score:.4f}")

Même si `paneeer` ou `masalaa` n'existent pas dans le corpus, FastText construit leur vecteur à partir de leurs **n-grammes de caractères** (sous-mots).
Leurs voisins restent donc proches de ceux du mot correct — c'est l'avantage principal de FastText sur Word2Vec, qui ne peut pas représenter un mot hors vocabulaire.

## Exercice 2 — Word2Vec (gensim) : CBOW et Skip-Gram

### Q1. Lecture du fichier et remplacement de `\n` par un espace
Le notebook lit `alice.txt` s'il existe, sinon `alice.docx`.

In [ ]:
from gensim.models import Word2Vec

if os.path.exists("alice.txt"):
    with open("alice.txt", encoding="utf-8") as f:
        texte = f.read()
else:
    import docx   # pip install python-docx
    texte = "\n".join(p.text for p in docx.Document("alice.docx").paragraphs)

texte = texte.replace("\n", " ")
print(len(texte), "caractères")
print(texte[:500])

### Q2. Découpage en phrases puis tokenisation en mots

In [ ]:
data = []
for phrase in sent_tokenize(texte):
    mots = [m.lower() for m in word_tokenize(phrase) if m.isalpha()]
    if mots:
        data.append(mots)

print("Nombre de phrases :", len(data))
print(data[:3])

### Q3. Modèles CBOW et Skip-Gram (`min_count=2`, `vector_size=100`, `window=5`)

In [ ]:
# sg=0 -> CBOW ; sg=1 -> Skip-Gram
model_cbow = Word2Vec(data, min_count=2, vector_size=100, window=5, sg=0)
model_sg   = Word2Vec(data, min_count=2, vector_size=100, window=5, sg=1)

print("Vocabulaire CBOW      :", len(model_cbow.wv))
print("Vocabulaire Skip-Gram :", len(model_sg.wv))

### Q4. Similarité cosinus entre trois mots

In [ ]:
paires = [("alice", "rabbit"), ("king", "queen"), ("cat", "mouse")]
for m1, m2 in paires:
    assert m1 in model_cbow.wv and m2 in model_cbow.wv, f"{m1} ou {m2} absent du vocabulaire"

resultats = []
for m1, m2 in paires:
    resultats.append({
        "Paire": f"{m1} / {m2}",
        "CBOW": model_cbow.wv.similarity(m1, m2),
        "Skip-Gram": model_sg.wv.similarity(m1, m2),
    })
pd.DataFrame(resultats)

### Q5. Mots les plus similaires à un mot choisi

In [ ]:
mot = "alice"
pd.DataFrame({
    "CBOW": [f"{w} ({s:.3f})" for w, s in model_cbow.wv.most_similar(mot, topn=10)],
    "Skip-Gram": [f"{w} ({s:.3f})" for w, s in model_sg.wv.most_similar(mot, topn=10)],
})

**Interprétation :** sur un petit corpus comme *Alice*, CBOW donne des similarités élevées et peu discriminantes (tous les mots fréquents se ressemblent),
alors que Skip-Gram produit des valeurs plus contrastées et des voisins plus pertinents pour les mots rares, car il apprend à partir de chaque paire (mot, contexte).